# Analysis of allowed requests per seconds and header extensions

In [ ]:
import openai
import sqlite3
import pandas as pd
import requests 
import time

# Initialize the client
client = openai.OpenAI(api_key= "xxxx")

assistant = client.beta.assistants.create(
    name="Text Processor",
    instructions="You are a text processor assistant. Always return a valid JSON String. Check if the text specifies a limit of requests per seonds, and if so add it to the returned json in key 'req_per_sec' as integer, if not that the field should be set to None. In addition, if there are requirements regarding a header which should be added/send along the requests, specify the header in the json at key 'header', if there is no such header mentioned, add None there. Ignore YWH-Alias, since this is not a header",
    tools=[{"type": "code_interpreter"}],
    model="gpt-4o-mini"
)

thread = client.beta.threads.create()


db_path = "../data/BugBountyData.sqlite"
query = f"SELECT * FROM rules WHERE rules IS NOT NULL"
conn = sqlite3.connect(db_path)
chunksize = 2500
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        message = client.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content=row['rules']
        )

        # Step 4: Run the Assistant
        run = client.beta.threads.runs.create(
            thread_id=thread.id,
            assistant_id=assistant.id,
        )

        while True:
            time.sleep(5)  
            run_status = client.beta.threads.runs.retrieve(
                thread_id=thread.id,
                run_id=run.id
            )
            if run_status.status == 'completed':
                messages = client.beta.threads.messages.list(
                    thread_id=thread.id
                )
                for msg in messages.data:
                    role = msg.role
                    content = msg.content[0].text.value
                    if role.capitalize() == "Assistant":
                        print(row['programHandle'])
                        print(f"{role.capitalize()}: {content}")
                
                break
        


ModuleNotFoundError: No module named 'openai'

# Extract contact Emails

In [ ]:
import openai
import sqlite3
import pandas as pd
import requests 
import time

# Initialize the client
client = openai.OpenAI(api_key= "xxxx")

assistant = client.beta.assistants.create(
    name="Text Processor",
    instructions="You are a text processor assistant. Always return a valid JSON String. Check if the text specifies a contact email, if so extract it to a list which should get returned.",
    tools=[{"type": "code_interpreter"}],
    model="gpt-4o-mini"
)

thread = client.beta.threads.create()


db_path = "../data/BugBountyData.sqlite"
query = f"SELECT * FROM rules WHERE rules IS NOT NULL"
conn = sqlite3.connect(db_path)
chunksize = 2500
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        message = client.beta.threads.messages.create(
            thread_id=thread.id,
            role="user",
            content=row['rules']
        )

        # Step 4: Run the Assistant
        run = client.beta.threads.runs.create(
            thread_id=thread.id,
            assistant_id=assistant.id,
        )

        while True:
            time.sleep(5)  
            run_status = client.beta.threads.runs.retrieve(
                thread_id=thread.id,
                run_id=run.id
            )
            if run_status.status == 'completed':
                messages = client.beta.threads.messages.list(
                    thread_id=thread.id
                )
                for msg in messages.data:
                    role = msg.role
                    content = msg.content[0].text.value
                    if role.capitalize() == "Assistant":
                        print(f"{role.capitalize()}: {content}")
                
                break
        


Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": [
    "security@doctolib.com"
  ]
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": [
    "security@doctolib.com"
  ]
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": [
    "security@doctolib.com"
  ]
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": [
    "security@doctolib.com"
  ]
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": []
}
```
Assistant: ```json
{
  "emails": [
    "security@doctolib.com"
  ]
}
```
Assis

KeyboardInterrupt: 

# Extract allowed and disallowed vulnerabilty types

In [ ]:
import openai
import sqlite3
import pandas as pd
import requests 
import time
import json
import re


###################################
# VulnType YesWeHack
# -> Easy since already unique entries for each in and out-of-scope vulnType
db_path = "../data/BugBountyData.sqlite"
query = f"SELECT * FROM vulnerabilityTypes WHERE vulnTypes IS NOT NULL and source='YesWeHack'"
conn = sqlite3.connect(db_path)
chunksize = 2500

vulnTypesYWH = {}
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        source = row['programHandle']
        if source not in vulnTypesYWH.keys():
            vulnTypesYWH[source] = {"inScope": "IN-SCOPES: ", "outOfScope": "OUT-OF-SCOPES: "}

        inscope = row['inScope']
        vulnType = row["vulnTypes"]
        if inscope:
            vulnTypesYWH[source]['inScope'] += "," + vulnType
        
        else:
            vulnTypesYWH[source]['outOfScope'] += "," + vulnType

#print("YesWeHack:")
#print(vulnTypesYWH)


###################################
# VulnType Intigriti
db_path = "../data/BugBountyData.sqlite"
query = f"SELECT * FROM vulnerabilityTypes WHERE vulnTypes IS NOT NULL and source='Intigriti'"
conn = sqlite3.connect(db_path)
chunksize = 2500

vulnTypesINTI = {}
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        source = row['programHandle']
        if source not in vulnTypesINTI.keys():
            vulnTypesINTI[source] = {"inScope": "IN-SCOPES: ", "outOfScope": "OUT-OF-SCOPES: "}

        inscope = row['inScope']
        vulnType = row["vulnTypes"]
        if inscope:
            vulnTypesINTI[source]['inScope'] += "," + vulnType
        
        else:
            vulnTypesINTI[source]['outOfScope'] += "," + vulnType

#print("Intigriti:")
#print(vulnTypesINTI)
   

####################################
# VulnTypes HackerOne
vulnTypesHO = {}
db_path = "../data/BugBountyData.sqlite"
query = f"SELECT * FROM rules WHERE rules IS NOT NULL and source='HackerOne'"
conn = sqlite3.connect(db_path)
chunksize = 2500
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        program_handle = row['programHandle']
        rule_value = row['rules']
        
        if program_handle not in vulnTypesHO:
            vulnTypesHO[program_handle] = {"rules": []}
        
        vulnTypesHO[program_handle]["rules"].append(rule_value)

#print("HackerOne:")
#print(vulnTypesHO)
        



####################################
# VulnTypes BugCrowd
vulnTypesBC = {}
db_path = "../data/BugBountyData.sqlite"
query = f"SELECT * FROM rules WHERE rules IS NOT NULL and source='BugCrowd'"
conn = sqlite3.connect(db_path)
chunksize = 2500
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        program_handle = row['programHandle']
        rule_value = row['rules']
        
        if program_handle not in vulnTypesBC:
            vulnTypesBC[program_handle] = {"rules": []}
        
        vulnTypesBC[program_handle]["rules"].append(rule_value)

#print("BugCrowd:")
#print(vulnTypesBC.keys())


####################################
# ChatGPT should clean up and aggreates the different types of in and out of scope vulnerabilities for each of the programs
# Cut it down to smaller lists

allVulnDesc = vulnTypesBC.copy()
allVulnDesc.update(vulnTypesHO)
allVulnDesc.update(vulnTypesINTI)
allVulnDesc.update(vulnTypesYWH)

print(len(list(allVulnDesc.items())))
full_list = list(vulnTypesINTI.items())


# Initialize the client
client = openai.OpenAI(api_key= "xxxx")
import json
import time

assistant = client.beta.assistants.create(
    name="Text Processor",
    instructions= """
    ###Question: Which vulnerability types are allowed to get tested (e.g. they are in-scope) and which are not allowed to get tested (outofscope)?
    
    ###Instruction: You are an assistant for data extraction, normalization and aggregation. Make sure to always use the same names for the same types of vulnerability types in your output, so normalize them. Remove any specifc URLs, company names, E-Mails etc. Always return a valid JSON in the below explained structure.
     
    ### Output Example: {'inscope': ['XSS', 'SQLI'], 'outOfScope': ['Self-XSS', 'CORS Misconfig', 'Recently disclosed new Vulnerabilities]""",
    tools=[{"type": "code_interpreter"}],
    model="gpt-4o-mini",
    timeout=30
)
print(f"[INFO] Created Assistant with ID: {assistant.id}")

thread = client.beta.threads.create()
print(f"[INFO] Created Thread with ID: {thread.id}")

progress = 0
vuln_type_details = {}
for program_name, descriptions_json in full_list:
    time.sleep(2)
    progress += 1
    if progress >= 50:
        print("[DEBUG] Reached 50 items, stopping early.")
        break

    print(f"[DEBUG] Processing program: {program_name} (#{progress}), data length: {len(descriptions_json['inScope']) + len(descriptions_json['outOfScope'])}")

    message = client.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=json.dumps(descriptions_json)
    )
    print(f"[DEBUG] Created message (ID: {message.id}) in thread {thread.id}")

    run = client.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id,
    )
    print(f"[INFO] Created run with ID: {run.id} for assistant {assistant.id}")

    while True:
        time.sleep(5)
        run_status = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id
        )
        print(f"[DEBUG] Current status of run {run.id}: {run_status.status}")

        if run_status.status == 'completed':
            messages = client.beta.threads.messages.list(thread_id=thread.id)
            assistant_messages = [m for m in messages.data if m.role.lower() == "assistant"]
            if assistant_messages:
                first_assistant_msg = assistant_messages[0]
                content = first_assistant_msg.content[0].text.value
                print(f"[INFO] Last Assistant message received: {content}")

                # Store the extracted data
                cleaned = re.sub(r'^```json\s*', '', content)
                cleaned = re.sub(r'```$', '', cleaned)
                extracted_data = json.loads(cleaned)
                vuln_type_details[program_name] = extracted_data

            break

        if run_status.status == 'failed':
            print("[ERROR] Run status:", run_status.status)
            print("[ERROR] Error info:", run_status.last_error)
            break

        if run_status.status == 'incomplete':
            print("[ERROR] Run status:", run_status.status)
            print("[ERROR] Error info:", run_status.last_error)
            break

print(vuln_type_details)
with open("extracted_vuln_types.json", "w", encoding="utf-8") as file:
    json.dump(vuln_type_details, file)


# Now correlate the data 
assistant = client.beta.assistants.create(
    name="Text Processor",
    instructions= """
    ###Question: Which vulnerability types are allowed how often to get tested (e.g. they are in-scope) and which are not allowed to get tested (outofscope)?
    
    ###Instruction: You are an assistant for data extraction, normalization and aggregation. Make sure to always use the same names for the same types of vulnerability types in your output, so normalize them. Remove any specifc URLs, company names, E-Mails etc. Increase the count at the input json at the specfic entry or add a new entry if a new vulnerability type is identfied. Later we have a JSON that hold the count for in and out of scope vulnerabilty types e.g. how often they occured.  Always return a valid JSON.
     
    ### Output Example: {'inscope': {'XSS': 2, 'SQLI':3}, 'outOfScope': {'Self-XSS': 4, 'CORS Misconfig': 5, 'Recently disclosed new Vulnerabilities': 10 }}""",
    tools=[{"type": "code_interpreter"}],
    model="gpt-4o-mini",
    timeout=30
)
print(f"[INFO] Created Assistant with ID: {assistant.id}")

thread = client.beta.threads.create()
print(f"[INFO] Created Thread with ID: {thread.id}")

progress = 0
vuln_json = {"inscope": {}, "outofscope": {}}
for program_name, vuln_type_json in vuln_type_details.items():
    progress += 1
    if progress >= 50:
        print("[DEBUG] Reached 50 items, stopping early.")
        break

    print(f"[DEBUG] Processing program: {program_name} (#{progress}), data length: {len(descriptions_json)}")

    message = client.beta.threads.messages.create(
        thread_id=thread.id,
        role="user",
        content=json.dumps({"json_overview": vuln_json, "new_text_to_analyse": vuln_type_json})
    )
    print(f"[DEBUG] Created message (ID: {message.id}) in thread {thread.id}")

    run = client.beta.threads.runs.create(
        thread_id=thread.id,
        assistant_id=assistant.id,
    )
    print(f"[INFO] Created run with ID: {run.id} for assistant {assistant.id}")

    while True:
        time.sleep(5)
        run_status = client.beta.threads.runs.retrieve(
            thread_id=thread.id,
            run_id=run.id
        )
        print(f"[DEBUG] Current status of run {run.id}: {run_status.status}")

        if run_status.status == 'completed':
            messages = client.beta.threads.messages.list(thread_id=thread.id)
            assistant_messages = [m for m in messages.data if m.role.lower() == "assistant"]
            if assistant_messages:
                first_assistant_msg = assistant_messages[0]
                content = first_assistant_msg.content[0].text.value
                print(f"[INFO] Last Assistant message received: {content}")

                # Store the extracted data
                cleaned = re.sub(r'^```json\s*', '', content)
                cleaned = re.sub(r'```$', '', cleaned)
                vuln_json = json.loads(cleaned)

            break

        if run_status.status == 'failed':
            print("[ERROR] Run status:", run_status.status)
            print("[ERROR] Error info:", run_status.last_error)
            break

# -------------- Remove all assistants ----------
assistants_list = client.beta.assistants.list()
print(f"[INFO] Found {len(assistants_list.data)} assistants; deleting them all now...")
for a in assistants_list.data:
    print(f"[DEBUG] Deleting assistant with ID: {a.id}")
    client.beta.assistants.delete(a.id)



625
[INFO] Created Assistant with ID: asst_k33ZOcvQhVhHIrYxmrBVQydC
[INFO] Created Thread with ID: thread_swp9QrN6Ek75m99xto0i0WFb
[DEBUG] Processing program: usz-vdp (#1), data length: 4610
[DEBUG] Created message (ID: msg_yFRR3c9lrySN41gF0iWBCEkn) in thread thread_swp9QrN6Ek75m99xto0i0WFb
[INFO] Created run with ID: run_EhfgrXaXmGfm2F1RAcPRTS5W for assistant asst_k33ZOcvQhVhHIrYxmrBVQydC
[DEBUG] Current status of run run_EhfgrXaXmGfm2F1RAcPRTS5W: in_progress
[DEBUG] Current status of run run_EhfgrXaXmGfm2F1RAcPRTS5W: in_progress
[DEBUG] Current status of run run_EhfgrXaXmGfm2F1RAcPRTS5W: in_progress
[DEBUG] Current status of run run_EhfgrXaXmGfm2F1RAcPRTS5W: in_progress
[DEBUG] Current status of run run_EhfgrXaXmGfm2F1RAcPRTS5W: in_progress
[DEBUG] Current status of run run_EhfgrXaXmGfm2F1RAcPRTS5W: completed
[INFO] Last Assistant message received: ```json
{
    "inscope": [],
    "outOfScope": [
        "API key disclosure without proven business impact",
        "Pre-Auth Account t

JSONDecodeError: Extra data: line 97 column 1 (char 7193)

In [ ]:
import openai
import sqlite3
import pandas as pd
import requests
import time
import json
import re

###################################
# 0) Setup OpenAI

openai.api_key = "YOUR_API_KEY_HERE"

# A helper function to query GPT with system + user messages
def query_gpt(system_instructions, user_content, model="gpt-4", timeout=30):
    """
    Sends a prompt to OpenAI ChatCompletion and returns the assistant's text response.
    """
    messages = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_content}
    ]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        request_timeout=timeout
    )
    return response.choices[0].message.content


###################################
# 1) Collect data from multiple sources

db_path = "../data/BugBountyData.sqlite"

# --- VulnType YesWeHack ---
query = "SELECT * FROM vulnerabilityTypes WHERE vulnTypes IS NOT NULL AND source='YesWeHack'"
conn = sqlite3.connect(db_path)
chunksize = 2500

vulnTypesYWH = {}
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        source = row['programHandle']
        if source not in vulnTypesYWH:
            vulnTypesYWH[source] = {"inScope": "IN-SCOPES: ", "outOfScope": "OUT-OF-SCOPES: "}
        inscope = row['inScope']
        vulnType = row["vulnTypes"]
        if inscope:
            vulnTypesYWH[source]['inScope'] += "," + vulnType
        else:
            vulnTypesYWH[source]['outOfScope'] += "," + vulnType

# --- VulnType Intigriti ---
query = "SELECT * FROM vulnerabilityTypes WHERE vulnTypes IS NOT NULL AND source='Intigriti'"
conn = sqlite3.connect(db_path)
chunksize = 2500

vulnTypesINTI = {}
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        source = row['programHandle']
        if source not in vulnTypesINTI:
            vulnTypesINTI[source] = {"inScope": "IN-SCOPES: ", "outOfScope": "OUT-OF-SCOPES: "}
        inscope = row['inScope']
        vulnType = row["vulnTypes"]
        if inscope:
            vulnTypesINTI[source]['inScope'] += "," + vulnType
        else:
            vulnTypesINTI[source]['outOfScope'] += "," + vulnType

# --- VulnTypes HackerOne ---
vulnTypesHO = {}
query = "SELECT * FROM rules WHERE rules IS NOT NULL AND source='HackerOne'"
conn = sqlite3.connect(db_path)
chunksize = 2500
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        program_handle = row['programHandle']
        rule_value = row['rules']
        if program_handle not in vulnTypesHO:
            vulnTypesHO[program_handle] = {"rules": []}
        vulnTypesHO[program_handle]["rules"].append(rule_value)

# --- VulnTypes BugCrowd ---
vulnTypesBC = {}
query = "SELECT * FROM rules WHERE rules IS NOT NULL AND source='BugCrowd'"
conn = sqlite3.connect(db_path)
chunksize = 2500
for df in pd.read_sql_query(query, conn, chunksize=chunksize):
    for _, row in df.iterrows():
        program_handle = row['programHandle']
        rule_value = row['rules']
        if program_handle not in vulnTypesBC:
            vulnTypesBC[program_handle] = {"rules": []}
        vulnTypesBC[program_handle]["rules"].append(rule_value)

conn.close()  # Done with DB

####################################
# 2) Combine Data

allVulnDesc = {}
allVulnDesc.update(vulnTypesBC)
allVulnDesc.update(vulnTypesHO)
allVulnDesc.update(vulnTypesINTI)
allVulnDesc.update(vulnTypesYWH)

print("[INFO] Combined total entries:", len(allVulnDesc))


####################################
# 3) First pass: Ask GPT to separate in-scope vs. out-of-scope

# We'll pick a small subset for demonstration
full_list = list(vulnTypesINTI.items())  # or any subset from allVulnDesc.items()

system_instructions_1 = """
You are an assistant for data extraction, normalization and aggregation.
###Question:
Which vulnerability types are allowed to get tested (e.g. they are in-scope)
and which are not allowed to get tested (outofscope)?

###Instruction:
Make sure to always use the same names for the same types of vulnerability types in your output, so normalize them.
Remove any specific URLs, company names, E-Mails, etc. Always return a valid JSON in this structure:
{"inscope": [...], "outOfScope": [...]}
"""

vuln_type_details = {}
progress = 0

for program_name, descriptions_json in full_list:
    progress += 1
    if progress > 3:  # limit for demonstration
        print("[DEBUG] Stopping early after 3 programs for demonstration.")
        break

    print(f"[INFO] Program {program_name} (# {progress}) → GPT prompt")
    user_content = json.dumps(descriptions_json)  # The raw data we want GPT to parse

    # Query GPT with a 30-second timeout
    gpt_response = query_gpt(system_instructions_1, user_content, model="gpt-4", timeout=30)
    print("[GPT RESPONSE]", gpt_response)

    # Attempt to parse the JSON
    # If GPT includes ```json ... ``` fences, remove them
    cleaned = re.sub(r'^```json\s*', '', gpt_response.strip())
    cleaned = re.sub(r'```$', '', cleaned.strip())

    try:
        extracted_data = json.loads(cleaned)
    except json.JSONDecodeError as e:
        print("[ERROR] JSON parsing failed. Response was:", gpt_response)
        extracted_data = {"inscope": [], "outOfScope": []}

    vuln_type_details[program_name] = extracted_data


####################################
# 4) Second pass: Correlate the data (counts, etc.)

# Suppose we want to build a structure like:
# {
#   "inscope": {"XSS": 2, "SQL Injection": 3, ...},
#   "outOfScope": {"Physical attacks": 10, ...}
# }

system_instructions_2 = """
You are an assistant for data extraction, normalization and aggregation.
###Question:
Which vulnerability types are allowed how often to get tested (in-scope) and
which are not allowed (outofscope)? We have a JSON that holds cumulative counts
for each type. If a new vulnerability type is found, add it with count=1; if it
exists, increment the count. Always return valid JSON in the form:
{
  "inscope": {"XSS": 2, "SQLI":3},
  "outOfScope": {"Self-XSS":4, "CORS Misconfig":5}
}
Remove any specific URLs, company names, E-Mails, etc.
"""

vuln_counts = {"inscope": {}, "outOfScope": {}}
progress = 0

for program_name, extracted_data in vuln_type_details.items():
    progress += 1
    if progress > 5:
        print("[DEBUG] Stopping early after 5 for demonstration.")
        break

    # We'll pass two parts:
    # - The existing "vuln_counts"
    # - The new extracted_data for this program
    message_obj = {
        "current_counts": vuln_counts,
        "new_vulns": extracted_data
    }

    user_content = json.dumps(message_obj)
    print(f"[INFO] Updating counts with {program_name} (# {progress})")
    gpt_response = query_gpt(system_instructions_2, user_content, model="gpt-4", timeout=30)
    print("[GPT RESPONSE]", gpt_response)

    # Clean up possible triple-backtick
    cleaned = re.sub(r'^```json\s*', '', gpt_response.strip())
    cleaned = re.sub(r'```$', '', cleaned.strip())

    try:
        updated_counts = json.loads(cleaned)
        vuln_counts = updated_counts
    except json.JSONDecodeError:
        print("[ERROR] JSON parsing failed for correlation step.")
        # keep old vuln_counts if parse fails

print("[INFO] Final aggregated vulnerability counts:")
print(json.dumps(vuln_counts, indent=2))


[INFO] Combined total entries: 628
[INFO] Program robinhoodbbp (# 1) → GPT prompt


APIRemovedInV1: 

You tried to access openai.ChatCompletion, but this is no longer supported in openai>=1.0.0 - see the README at https://github.com/openai/openai-python for the API.

You can run `openai migrate` to automatically upgrade your codebase to use the 1.0.0 interface. 

Alternatively, you can pin your installation to the old version, e.g. `pip install openai==0.28`

A detailed migration guide is available here: https://github.com/openai/openai-python/discussions/742
